# Day 49 — PCA, SVD and t-SNE

Dimensionality reduction takes d columns and hands back two, and the d − 2 that went
away leave no trace. What comes out is a picture, and a picture is extremely easy to
over-read.

```
PCA   answers: which directions carry the most variance?
t-SNE answers: which points are each other's neighbours?
```

Neither of those is *which directions matter* or *how far apart are these groups*, and
those are the two things people read off the plot. Everything below is a measurement of
the gap between what was optimised and what was read: a first component that points at
where the data sits rather than how it is shaped, a component holding 98% of the
variance and none of the signal, a covariance matrix that rounds a real direction out of
existence, and a t-SNE map that draws three clusters in pure noise and then draws
different ones when you change the seed.

## PCA is the SVD of the centred data

Centre, take the SVD, and the right singular vectors are the principal directions. The
covariance route — build XᵀX / (n−1) and eigendecompose it — is the same mathematics and
a worse algorithm; the section on conditioning measures how much worse.

Note that `np.linalg.eigh` returns eigenvalues in **ascending** order, so the covariance
route needs an explicit reversal. Forget it and you get a perfectly ordinary looking
answer pointing at the least important direction.

In [1]:
class PCA:
    """Principal components, computed from the SVD of the centred data.

    The covariance route (eigendecompose Xc^T Xc / (n - 1) on the centred data) gives
    the same answer in exact arithmetic and a worse one in floating point - section 6
    measures how much worse.
    """

    def __init__(self, k=None, center_data=True):
        self.k = k
        self.center_data = center_data

    def fit(self, X):
        self.mean_ = X.mean(axis=0) if self.center_data else np.zeros(X.shape[1])
        Xc = X - self.mean_
        U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
        self.singular_values_ = s
        self.components_ = Vt                       # rows are the directions
        # the variance along component j, with the same 1/(n-1) convention as np.var(ddof=1)
        self.explained_variance_ = s ** 2 / (len(X) - 1)
        tot = self.explained_variance_.sum()
        self.explained_variance_ratio_ = self.explained_variance_ / tot if tot > 0 else self.explained_variance_
        return self

    def transform(self, X, k=None):
        k = k or self.k or self.components_.shape[0]
        return (X - self.mean_) @ self.components_[:k].T

    def inverse_transform(self, Z):
        return Z @ self.components_[:Z.shape[1]] + self.mean_


def pca_via_covariance(X, center_data=True):
    """The textbook route: build the covariance matrix and eigendecompose it.

    `np.linalg.eigh` returns eigenvalues in ascending order, so everything has to be
    reversed - a step that is easy to forget and that produces a perfectly ordinary
    looking answer pointing at the least important direction.
    """
    Xc = X - X.mean(axis=0) if center_data else X
    C = Xc.T @ Xc / (len(X) - 1)
    w, V = np.linalg.eigh(C)
    order = np.argsort(w)[::-1]
    return w[order], V[:, order].T


# ================================================================ 4. data sets
def anisotropic(n=300, seed=0):
    """A stretched cloud sitting a long way from the origin.  The offset is bigger
    than the spread, which is the only thing section 3 needs."""
    rng = np.random.default_rng(seed)
    A = np.array([[2.0, 0.5], [0.5, 0.6]])
    return rng.standard_normal((n, 2)) @ A + np.array([8.0, 5.0])


import math
import numpy as np
np.set_printoptions(precision=4, suppress=True)

X = anisotropic(300, seed=0)
p = PCA().fit(X)
w, V = pca_via_covariance(X)
print('singular values        ', p.singular_values_)
print('explained variance     ', p.explained_variance_)
print('covariance eigenvalues ', w)
print('the two routes agree to', f'{np.abs(p.explained_variance_ - w).max():.3e}')
print()
print('PC1 from the SVD ', p.components_[0])
print('PC1 from eigh    ', V[0])
print('dot product      ', f'{p.components_[0] @ V[0]:+.6f}')
print()
print('With a simple eigenvalue an eigenvector is defined only up to its sign:')
print('v and -v span the same line, so two libraries can hand you flipped plots and')
print('neither is wrong.  Compare components with |u . v|; pin a convention for plots.')

singular values         [37.6426  7.4918]
explained variance      [4.739  0.1877]
covariance eigenvalues  [4.739  0.1877]
the two routes agree to 4.441e-15

PC1 from the SVD  [-0.9521 -0.3059]
PC1 from eigh     [-0.9521 -0.3059]
dot product       +1.000000

With a simple eigenvalue an eigenvector is defined only up to its sign:
v and -v span the same line, so two libraries can hand you flipped plots and
neither is wrong.  Compare components with |u . v|; pin a convention for plots.


## Silent failure #1: forgetting to subtract the mean

Without centring, "variance" is measured about the origin instead of about the data, so
the first component points at **where the data is** rather than at how it is shaped.

The dangerous part is the score. The broken version reports a *higher* explained-variance
ratio, because the offset it is now measuring is bigger than the spread it should have
been measuring — the denominator was inflated along with the numerator. The number
anyone actually reports goes **up** when you make this mistake. Not every number is
fooled, which is the useful part: reconstruction error gets worse and does catch it. It
is simply not the number that ends up on the slide.

In [2]:
pc = PCA(center_data=True).fit(X)
pu = PCA(center_data=False).fit(X)
mean_dir = X.mean(axis=0) / np.linalg.norm(X.mean(axis=0))
ang = lambda a, b: math.degrees(math.acos(min(1.0, abs(float(a @ b)))))
print('the cloud sits at', np.round(X.mean(axis=0), 3))
print()
print('centred   PC1', pc.components_[0], ' explained variance ratio', pc.explained_variance_ratio_)
print('uncentred PC1', pu.components_[0], ' explained variance ratio', pu.explained_variance_ratio_)
print()
print(f'uncentred PC1 vs the direction of the mean: {ang(pu.components_[0], mean_dir):.3f} deg')
print(f'uncentred PC1 vs the real PC1             : {ang(pu.components_[0], pc.components_[0]):.3f} deg')

the cloud sits at [7.85  4.981]

centred   PC1 [-0.9521 -0.3059]  explained variance ratio [0.9619 0.0381]
uncentred PC1 [-0.8508 -0.5254]  explained variance ratio [0.995 0.005]

uncentred PC1 vs the direction of the mean: 0.701 deg
uncentred PC1 vs the real PC1             : 13.885 deg


## Silent failure #2: the units pick the axes

Day 47 and Day 48 both ran into this. PCA is where it is most visible, because the answer
*is* a direction. Two independent noise columns, one in the hundreds and one in [0, 1],
and PCA reports that a single component explains essentially all of the variance.

That number is true. It is a fact about the units, not about the data.

In [3]:
def requests(n=300, seed=3):
    """Day 47's two columns again: prompt_tokens in the hundreds, a hit rate in [0, 1]."""
    rng = np.random.default_rng(seed)
    return np.column_stack([rng.uniform(200, 4000, n), rng.uniform(0, 1, n)])


R = requests(300, seed=3)
pr = PCA().fit(R)
Rz = (R - R.mean(axis=0)) / R.std(axis=0)
pz = PCA().fit(Rz)
print(f'raw units    : PC1 [{pr.components_[0][0]:.8f} {pr.components_[0][1]:.8f}]')
print(f'               explained variance ratio {pr.explained_variance_ratio_[0]:.8f}')
print(f'standardised : PC1 [{pz.components_[0][0]:.8f} {pz.components_[0][1]:.8f}]')
print(f'               explained variance ratio {pz.explained_variance_ratio_[0]:.8f}')

raw units    : PC1 [1.00000000 0.00003620]
               explained variance ratio 0.99999992
standardised : PC1 [0.70710678 0.70710678]
               explained variance ratio 0.56328964


## Silent failure #3: explained variance is not usefulness

Two classes separated along one direction, with a far larger amount of variance along a
direction that carries no information, and the whole thing rotated so that neither is an
axis.

PCA finds the right *pair* of directions — the rotation is correct — and then sorts them
by the wrong key. "Keep 95% of the variance" discards the only component that could
answer the question. PCA never sees the labels, so it cannot know.

In [4]:
def hidden_signal(n=400, seed=1, sep=0.8, noise_scale=6.0, angle=30.0):
    """Two classes separated along one direction, with a much larger amount of
    variance along a direction that carries no information at all.  Then the whole
    thing is rotated, so the useful direction is not an axis."""
    rng = np.random.default_rng(seed)
    y = rng.integers(0, 2, n)
    signal = np.where(y == 1, sep, -sep) + 0.25 * rng.standard_normal(n)
    noise = noise_scale * rng.standard_normal(n)
    X = np.column_stack([noise, signal])
    t = math.radians(angle)
    R = np.array([[math.cos(t), -math.sin(t)], [math.sin(t), math.cos(t)]])
    return X @ R.T, y


def best_threshold_accuracy(v, y):
    """The best accuracy any single cut on one axis can reach.  A cheap, honest way
    to ask whether a projection kept the thing you cared about."""
    order = np.argsort(v)
    ys = y[order]
    n = len(ys)
    ones = np.concatenate([[0], np.cumsum(ys)])
    best = 0.0
    for i in range(n + 1):
        left_zeros = i - ones[i]
        right_ones = ones[n] - ones[i]
        best = max(best, (left_zeros + right_ones) / n, (ones[i] + (n - i) - right_ones) / n)
    return float(best)


H, hy = hidden_signal(400, seed=1)
ph = PCA().fit(H)
Z2 = ph.transform(H, 2)
per_comp = [best_threshold_accuracy(Z2[:, j], hy) for j in range(2)]
print(f'{"component":>10} {"variance explained":>19} {"best single cut":>17}')
for j in range(2):
    print(f'{"PC" + str(j + 1):>10} {100 * ph.explained_variance_ratio_[j]:>18.2f}% '
          f'{per_comp[j]:>17.4f}')
print()
print(f'the variance ratio between the two is {ph.explained_variance_[0] / ph.explained_variance_[1]:.1f} to 1')

 component  variance explained   best single cut
       PC1              98.33%            0.5450
       PC2               1.67%            1.0000

the variance ratio between the two is 59.0 to 1


## Why the SVD and not the covariance matrix

Forming XᵀX squares the condition number. `float64` carries about 16 digits, so once
cond(X) passes roughly 1e8 the small singular values of the squared problem are below the
noise floor — and come back as exactly zero.

There is a second symptom in the same place: `eigh` returns a small **negative**
eigenvalue for a covariance matrix, which is a negative variance. `np.sqrt` of it is
`nan`, so code that takes standard deviations from eigenvalues has to clamp at zero, and
clamping is what hides the problem.

In [5]:
def ill_conditioned(n=200, eps=1e-8, seed=0):
    """Singular values proportional to 1, eps, eps/2 in a random orthogonal basis, so
    the right answer is available to compare against.  The basis is scaled by
    sqrt(n - 1) so the covariance route's eigenvalues come out near 1 - that scale is
    why the smallest singular value below reads 7.04e-04 and not 5e-05 for eps=1e-4."""
    rng = np.random.default_rng(seed)
    Q = np.linalg.qr(rng.standard_normal((3, 3)))[0]
    base = np.linalg.qr(rng.standard_normal((n, 3)))[0] * math.sqrt(n - 1)
    return base @ np.diag([1.0, eps, eps / 2]) @ Q.T


print(f'{"eps":>8} {"cond(X)":>10} {"cond(cov)":>11} {"from SVD":>12} {"from eigh":>12} {"rel. error":>11}')
for eps in (1e-4, 1e-6, 1e-8, 1e-9):
    Xi = ill_conditioned(200, eps=eps, seed=0)
    Xc = Xi - Xi.mean(axis=0)
    sv = np.linalg.svd(Xc, full_matrices=False)[1]
    wi, _ = pca_via_covariance(Xi)
    sv_eig = np.sqrt(np.maximum(wi, 0) * (len(Xi) - 1))
    # cond(cov) is measured from the covariance matrix, not squared by hand
    cov_cond = float(np.linalg.cond(Xc.T @ Xc / (len(Xi) - 1)))
    print(f'{eps:>8.0e} {sv[0] / sv[-1]:>10.1e} {cov_cond:>11.1e} '
          f'{sv[-1]:>12.4e} {sv_eig[-1]:>12.4e} {abs(sv_eig[-1] - sv[-1]) / sv[-1]:>11.3e}')
wi, _ = pca_via_covariance(ill_conditioned(200, eps=1e-8, seed=0))
print()
print('smallest eigenvalue of a covariance matrix:', f'{wi.min():.3e}', '<- a negative variance')

     eps    cond(X)   cond(cov)     from SVD    from eigh  rel. error
   1e-04    2.0e+04     4.0e+08   7.0425e-04   7.0425e-04   4.582e-08
   1e-06    2.0e+06     4.0e+12   7.0425e-06   7.0416e-06   1.197e-04
   1e-08    2.0e+08     4.2e+16   7.0425e-08   0.0000e+00   1.000e+00
   1e-09    2.0e+09     6.5e+17   7.0425e-09   0.0000e+00   1.000e+00

smallest eigenvalue of a covariance matrix: -1.435e-16 <- a negative variance


## Truncated SVD and what "explained variance" actually means

Eckart–Young: truncating the SVD at k is the best rank-k approximation in Frobenius norm,
and the error is exactly the square root of the sum of the discarded singular values
squared. The last column below is the share of the squared Frobenius norm kept, and on
**centred** data that is exactly PCA's explained-variance ratio — the same ratio, one
step before the statistics. (`A` here is not centred, so the two differ slightly.)

That identity is all "explained variance" ever means, which is why the phrase is so easy
to misuse. It is a statement about reconstructing the matrix, and nothing at all about
reconstructing the thing you wanted.

In [6]:
rng = np.random.default_rng(5)
A = rng.standard_normal((120, 40)) @ rng.standard_normal((40, 40))
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print(f'{"k":>4} {"Frobenius error":>16} {"sqrt(sum dropped^2)":>21} {"|A|_F^2 kept":>14}')
for k in (1, 5, 10, 20, 40):
    Ak = (U[:, :k] * s[:k]) @ Vt[:k]
    print(f'{k:>4} {float(np.linalg.norm(A - Ak)):>16.6f} '
          f'{math.sqrt((s[k:] ** 2).sum()):>21.6f} '
          f'{(s[:k] ** 2).sum() / (s ** 2).sum():>14.4f}')

   k  Frobenius error   sqrt(sum dropped^2)   |A|_F^2 kept
   1       425.474593            425.474593         0.1185
   5       338.494196            338.494196         0.4421
  10       248.372324            248.372324         0.6996
  20       122.977998            122.977998         0.9264
  40         0.000000              0.000000         1.0000


## t-SNE: keep the neighbours, let go of everything else

**P**: in the original space, turn distances into neighbour probabilities with a
gaussian. Every row gets its *own* width, found by binary search so that the perplexity
of that row hits the number you asked for (to a tolerance of 1e-5, so perplexity 30
comes out between 29.9997 and 30.0003). Perplexity is 2^entropy — "how many neighbours
does this point effectively have".

**Q**: in the 2-D map, use a Student-t with one degree of freedom instead. Its tail is
heavy, so a pair that is moderately far apart in the original space is allowed to be very
far apart on the page. That is what stops everything piling into the middle.

Then minimise KL(P ‖ Q) by moving the points. The embedding *is* the parameter vector:
there is no function being fitted, which is why there is no `transform()` for new points
and why two runs are two different pictures.

In [7]:
# ================================================================ 2. t-SNE
def pairwise_sq_dists(X):
    """|a - b|^2 for every pair.  There are n(n-1)/2 of them, which is the whole
    reason t-SNE is slow and the whole point of LeetCode 719 below."""
    x2 = np.einsum('ij,ij->i', X, X)
    return np.maximum(x2[:, None] - 2.0 * X @ X.T + x2[None, :], 0.0)


def conditional_P(D, perplexity, tol=1e-5, max_iter=60):
    """Row i gets its own gaussian width, chosen by binary search so that the
    perplexity of row i's neighbour distribution hits the number you asked for, to
    within `tol` in nats (the search stops at 1e-5, i.e. perplexity 30 comes out
    somewhere in 29.9997-30.0003, not at exactly 30).

    Perplexity is 2^entropy: "how many neighbours does this point effectively have".
    Every row therefore gets a different sigma, which is how t-SNE copes with regions
    of different density - and also why the output says nothing about density.
    """
    n = len(D)
    P = np.zeros((n, n))
    target = math.log(perplexity)
    betas = np.zeros(n)
    for i in range(n):
        lo, hi = -np.inf, np.inf
        beta = 1.0                                  # beta = 1 / (2 sigma^2)
        Di = np.delete(D[i], i)
        for _ in range(max_iter):
            Pi = np.exp(-Di * beta)
            s = Pi.sum()
            if s <= 0:
                H, Pi = 0.0, np.zeros_like(Di)
            else:
                H = math.log(s) + beta * float(np.sum(Di * Pi)) / s
                Pi = Pi / s
            if abs(H - target) < tol:
                break
            if H > target:                          # too many neighbours: narrow it
                lo = beta
                beta = beta * 2 if hi == np.inf else (beta + hi) / 2
            else:
                hi = beta
                beta = beta / 2 if lo == -np.inf else (beta + lo) / 2
        betas[i] = beta
        P[i] = np.insert(Pi, i, 0.0)
    return P, betas


def joint_P(X, perplexity):
    P, betas = conditional_P(pairwise_sq_dists(X), perplexity)
    P = (P + P.T) / (2 * len(X))                    # symmetrise, then normalise
    # the floor keeps log(P / Q) finite; renormalise so P is still a distribution
    P = np.maximum(P, 1e-12)
    return P / P.sum(), betas


def tsne(X, perplexity=30.0, dim=2, n_iter=600, lr=200.0, seed=0,
         early_exaggeration=12.0, exaggerate_until=100, momentum=(0.5, 0.8)):
    """Minimise KL(P || Q) by gradient descent on the embedding itself.

    The embedding IS the parameter vector - there is no function being fitted, which
    is why t-SNE has no `transform` for new points and why two runs differ.
    """
    rng = np.random.default_rng(seed)
    n = len(X)
    P, _ = joint_P(X, perplexity)
    Y = rng.standard_normal((n, dim)) * 1e-4
    dY = np.zeros_like(Y)
    gains = np.ones_like(Y)
    losses = []
    for it in range(n_iter):
        Pe = P * early_exaggeration if it < exaggerate_until else P
        num = 1.0 / (1.0 + pairwise_sq_dists(Y))    # the Student-t kernel
        np.fill_diagonal(num, 0.0)
        Q = np.maximum(num / num.sum(), 1e-12)
        PQ = Pe - Q
        # dC/dy_i = 4 sum_j (p_ij - q_ij) (y_i - y_j) / (1 + |y_i - y_j|^2)
        W = PQ * num
        grad = 4.0 * (Y * W.sum(axis=1)[:, None] - W @ Y)
        mom = momentum[0] if it < 250 else momentum[1]
        gains = np.where(np.sign(grad) != np.sign(dY), gains + 0.2, gains * 0.8)
        gains = np.maximum(gains, 0.01)
        dY = mom * dY - lr * gains * grad
        Y = Y + dY
        Y = Y - Y.mean(axis=0)
        losses.append(float(np.sum(P * np.log(P / Q))))
    return Y, losses


def neighbour_recall(X, Y, k=10):
    """Of the k nearest neighbours a point has in the original space, how many are
    still among its k nearest in the embedding?  This is the thing t-SNE optimises,
    and the only thing the picture is entitled to claim."""
    def knn(A):
        D = pairwise_sq_dists(A)
        np.fill_diagonal(D, np.inf)
        return np.argsort(D, axis=1)[:, :k]
    a, b = knn(X), knn(Y)
    return float(np.mean([len(set(a[i]) & set(b[i])) / k for i in range(len(X))]))


def blobs(spreads, centers, n_per=60, seed=1):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for i, (c, s) in enumerate(zip(centers, spreads)):
        X.append(np.array(c, dtype=float) + s * rng.standard_normal((n_per, len(c))))
        y += [i] * n_per
    return np.vstack(X), np.array(y)


def spread(X):
    return float(np.sqrt(((X - X.mean(axis=0)) ** 2).sum(axis=1).mean()))


def centre_distance(X, a, b):
    return float(np.linalg.norm(X[a].mean(axis=0) - X[b].mean(axis=0)))


def pairwise_corr(A, B):
    """Correlation between the two sets of pairwise distances - how much two pictures
    of the same points agree."""
    iu = np.triu_indices(len(A), 1)
    return float(np.corrcoef(np.sqrt(pairwise_sq_dists(A))[iu],
                             np.sqrt(pairwise_sq_dists(B))[iu])[0, 1])


Xb, yb = blobs([1.0, 1.0, 1.0], [[0, 0, 0, 0, 0], [15, 0, 0, 0, 0], [0, 40, 0, 0, 0]],
               60, seed=1)
P, betas = joint_P(Xb, 30.0)
sig = 1.0 / np.sqrt(2.0 * betas)
print(f'perplexity 30 gives each row its own width: sigma from {sig.min():.4f} to {sig.max():.4f}')
print(f'P sums to {P.sum():.6f} and is symmetric to {np.abs(P - P.T).max():.3e}')
Yb, loss = tsne(Xb, perplexity=30.0, n_iter=600, lr=50.0, seed=0)
print()
print(f'600 iterations: KL {loss[0]:.4f} -> {loss[-1]:.4f}')
print(f'neighbours kept (10 nearest, data vs map): {neighbour_recall(Xb, Yb, 10):.4f}')
print('That recall number is the only claim the picture is entitled to make.')

perplexity 30 gives each row its own width: sigma from 0.7827 to 1.7242
P sums to 1.000000 and is symmetric to 0.000e+00

600 iterations: KL 1.6949 -> 0.2499
neighbours kept (10 nearest, data vs map): 0.6206
That recall number is the only claim the picture is entitled to make.


## Silent failure #4: t-SNE draws clusters in pure noise

180 points of 10-dimensional gaussian noise. There is nothing in it. A silhouette near 0
means "this grouping is arbitrary"; the data scores about 0.09 and the four pictures of
it score between 4.2 and 4.6 times that.

This is Day 47's k-means on uniform noise, one layer further along: the method is built
to produce groups, so it produces groups. The test that catches it costs one more run —
real structure survives a different random seed, invented structure does not.

In [8]:
# ================================================================ 3. reading the picture
def kmeans_small(X, k, seed=0, restarts=8, iters=60):
    """A small Lloyd loop with restarts, only used to ask how clustered a picture
    looks.  Day 47 has the real one, and all of its caveats."""
    best = None
    for r in range(restarts):
        rng = np.random.default_rng(seed + r)
        C = X[rng.choice(len(X), k, replace=False)].copy()
        lab = np.zeros(len(X), dtype=int)
        for _ in range(iters):
            lab = np.argmin(((X[:, None, :] - C[None]) ** 2).sum(axis=2), axis=1)
            new = np.array([X[lab == j].mean(axis=0) if np.any(lab == j) else C[j]
                            for j in range(k)])
            if np.allclose(new, C):
                break
            C = new
        inertia = float(np.sum((X - C[lab]) ** 2))
        if best is None or inertia < best[0]:
            best = (inertia, lab)
    return best[1]


def silhouette(X, labels):
    """Mean over points of (b - a) / max(a, b): how much closer a point is to its own
    group than to the nearest other one.  Near 0 means the grouping is arbitrary."""
    D = np.sqrt(pairwise_sq_dists(X))
    ks = np.unique(labels)
    out = []
    for i in range(len(X)):
        own = labels[i] == labels
        own[i] = False
        if not own.any():
            continue
        a = D[i, own].mean()
        b = min(D[i, labels == c].mean() for c in ks if c != labels[i])
        out.append((b - a) / max(a, b))
    return float(np.mean(out))


def pure_noise(n=180, d=10, seed=7):
    return np.random.default_rng(seed).standard_normal((n, d))


Xn = pure_noise(180, 10, seed=7)
s_raw = silhouette(Xn, kmeans_small(Xn, 3, seed=0))
print(f'{"perplexity":>11} {"silhouette of the data":>24} {"silhouette of the picture":>27}')
for per in (5, 15, 30, 50):
    Yn, _ = tsne(Xn, perplexity=float(per), n_iter=600, lr=50.0, seed=0)
    print(f'{per:>11} {s_raw:>24.4f} {silhouette(Yn, kmeans_small(Yn, 3, seed=0)):>27.4f}')
print()
for nm, data in [('three real blobs', Xb), ('pure 10-D noise', Xn)]:
    Ys = [tsne(data, perplexity=30.0, n_iter=600, lr=50.0, seed=sd)[0] for sd in range(3)]
    cs = [pairwise_corr(Ys[a], Ys[b]) for a, b in ((0, 1), (0, 2), (1, 2))]
    print(f'{nm:>17}: correlation between three runs {np.round(cs, 4).tolist()}, '
          f'mean {np.mean(cs):.4f}')

 perplexity   silhouette of the data   silhouette of the picture
          5                   0.0872                      0.3972
         15                   0.0872                      0.3763
         30                   0.0872                      0.3707
         50                   0.0872                      0.3689

 three real blobs: correlation between three runs [0.9871, 0.9894, 0.99], mean 0.9889
  pure 10-D noise: correlation between three runs [0.3244, 0.2875, 0.3439], mean 0.3186


## Silent failure #5: sizes and distances on the page mean nothing

A cluster ten times wider comes out the same size as the others, because every row chose
its own sigma to reach the same perplexity. That per-row width is how t-SNE copes with
regions of different density, and it is exactly why the output cannot report density.

And the between-cluster distances are simply gone. There is no bug here: KL(P ‖ Q)
contains no term that asks about the gap between two clusters, so nothing in the
optimisation was ever trying to get that right.

In [9]:
Xs, ys = blobs([0.5, 5.0, 0.5], [[0, 0, 0, 0, 0], [20, 0, 0, 0, 0], [0, 60, 0, 0, 0]],
               60, seed=1)
Ys_, _ = tsne(Xs, perplexity=30.0, n_iter=600, lr=50.0, seed=0)
m = [ys == i for i in range(3)]
o_sp = [spread(Xs[mi]) for mi in m]
e_sp = [spread(Ys_[mi]) for mi in m]
print(f'spreads in the data : {np.round(o_sp, 3).tolist()}   widest / narrowest = {max(o_sp) / min(o_sp):.2f}')
print(f'spreads in the map  : {np.round(e_sp, 3).tolist()}   widest / narrowest = {max(e_sp) / min(e_sp):.2f}')
print()
print(f'{"true distance ratio":>21} {"ratio on the page":>20}')
for far in (30, 60, 120, 400):
    Xf, yf = blobs([1.0, 1.0, 1.0], [[0, 0, 0, 0, 0], [15, 0, 0, 0, 0], [0, far, 0, 0, 0]],
                   60, seed=1)
    Yf, _ = tsne(Xf, perplexity=30.0, n_iter=600, lr=50.0, seed=0)
    mf = [yf == i for i in range(3)]
    print(f'{centre_distance(Xf, mf[0], mf[2]) / centre_distance(Xf, mf[0], mf[1]):>21.2f} '
          f'{centre_distance(Yf, mf[0], mf[2]) / centre_distance(Yf, mf[0], mf[1]):>20.2f}')

spreads in the data : [1.025, 10.818, 1.182]   widest / narrowest = 10.56
spreads in the map  : [2.159, 2.421, 2.155]   widest / narrowest = 1.12

  true distance ratio    ratio on the page
                 2.02                 1.04
                 4.03                 1.04
                 8.05                 1.04
                26.82                 1.02


## Silent failure #6: the loss gets better while the map gets worse

P is rebuilt from scratch for every perplexity, so each row below is a **different
objective function**. KL values are not comparable across perplexities, and the one
number the optimiser prints is therefore useless for choosing the one knob that matters.

In [10]:
print(f'{"perplexity":>11} {"final KL":>10} {"neighbours kept":>17}')
for per in (2, 5, 30, 80, 150):
    Yp, lp = tsne(Xb, perplexity=float(per), n_iter=600, lr=50.0, seed=0)
    print(f'{per:>11} {lp[-1]:>10.4f} {neighbour_recall(Xb, Yp, 10):>17.4f}')

 perplexity   final KL   neighbours kept
          2     0.6008            0.5283
          5     0.5849            0.5822
         30     0.2499            0.6206
         80     0.0010            0.5028
        150     0.0025            0.3656


## LeetCode 719 and LeetCode 2352

Both halves of this day are built on pairwise distances, and there are n(n−1)/2 of them —
at n = 10,000 that is 49,995,000, the number you are not allowed to materialise. **719**
is that constraint isolated: binary search the *answer* and count with a sliding window,
because `count(m) = how many pairs are at most m apart` is non-decreasing in m.

**2352** is about Aᵀ, the other thing this day is made of. A row and a column are equal
when they are the same *sequence*, so the key has to be the sequence. A tuple is; a
string of concatenated digits is not — `[1, 23]` and `[12, 3]` both become `"123"`.

In [11]:
# ================================================================ 5. LeetCode
class SolutionKthPairDistance:
    def smallestDistancePair(self, nums: list[int], k: int) -> int:
        # There are n(n-1)/2 distances and up to 5e7 of them, so they are never
        # built.  Binary search the ANSWER instead: count(m) = how many pairs are
        # at most m apart is non-decreasing in m, so the smallest m with
        # count(m) >= k is the k-th smallest distance.
        nums.sort()
        n = len(nums)

        def count(m: int) -> int:
            total = j = 0
            for i in range(n):
                while nums[i] - nums[j] > m:
                    j += 1
                total += i - j          # every index in [j, i) pairs with i
            return total

        lo, hi = 0, nums[-1] - nums[0]
        while lo < hi:
            mid = (lo + hi) // 2
            if count(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo


def kth_pair_distance_lo1(nums, k):
    """The same search started at lo = 1 instead of lo = 0 - "a distance is at least
    1" - which is true of every pair of DIFFERENT values and false the moment the
    array repeats one."""
    nums = sorted(nums)
    n = len(nums)

    def count(m):
        total = j = 0
        for i in range(n):
            while nums[i] - nums[j] > m:
                j += 1
            total += i - j
        return total

    lo, hi = 1, nums[-1] - nums[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return max(lo, 0)


def kth_pair_distance_brute(nums, k):
    ds = sorted(abs(a - b) for i, a in enumerate(nums) for b in nums[i + 1:])
    return ds[k - 1]


class SolutionEqualPairs:
    def equalPairs(self, grid: list[list[int]]) -> int:
        # A row and a column are equal when they are the same sequence of numbers.
        # A tuple keeps the element boundaries; gluing the digits into one string
        # throws them away, so (1, 2, 34) and (1, 23, 4) collide.
        from collections import Counter
        rows = Counter(tuple(r) for r in grid)
        return sum(rows[col] for col in zip(*grid))


def equal_pairs_joined(grid):
    """The same count with `''.join(map(str, row))` as the key: [1, 23] and [12, 3]
    become the same string, so distinct rows are merged and the count comes out high."""
    from collections import Counter
    rows = Counter(''.join(map(str, r)) for r in grid)
    return sum(rows[''.join(map(str, c))] for c in zip(*grid))


def equal_pairs_brute(grid):
    n = len(grid)
    return sum(1 for r in range(n) for c in range(n)
               if all(grid[r][i] == grid[i][c] for i in range(n)))


sol = SolutionKthPairDistance()
print('[1, 3, 1] k=1 ->', sol.smallestDistancePair([1, 3, 1], 1), '(distances 0, 2, 2)')
print('[1, 1, 1] k=2 ->', sol.smallestDistancePair([1, 1, 1], 2))
print('[1, 6, 1] k=3 ->', sol.smallestDistancePair([1, 6, 1], 3))
rng719 = np.random.default_rng(5)
bad, first = 0, None
for _ in range(5000):
    n = int(rng719.integers(2, 9))
    a = [int(v) for v in rng719.integers(0, 15, n)]
    k = int(rng719.integers(1, n * (n - 1) // 2 + 1))
    want = kth_pair_distance_brute(a, k)
    got = kth_pair_distance_lo1(a[:], k)
    if got != want:
        bad += 1
        if first is None:
            first = (sorted(a), k, got, want)
print()
print(f'starting the search at lo = 1 differs on {bad}/5000 cases ({100 * bad / 5000:.1f}%)')
print(f'  first one: nums={first[0]}, k={first[1]} -> {first[2]} instead of {first[3]}')
print()
sp2 = SolutionEqualPairs()
print('[[3,2,1],[1,7,6],[2,7,7]] ->', sp2.equalPairs([[3, 2, 1], [1, 7, 6], [2, 7, 7]]))
trap = [[1, 2, 34], [23, 5, 6], [4, 7, 8]]
print(f'{trap}: correct {sp2.equalPairs(trap)}, brute {equal_pairs_brute(trap)}, '
      f'joined-string key {equal_pairs_joined(trap)}')
from collections import Counter
pairs = [(a, b) for a in range(1, 100) for b in range(1, 100)]
keys = Counter(''.join(map(str, s)) for s in pairs)
shared = [k for k, v in keys.items() if v > 1]
assert all(keys[k] == 2 for k in shared)   # every clash here is an exact pair
nonuniq = sum(keys[k] for k in shared)
print(f'of {len(pairs)} two-element sequences with values in 1..99, {len(keys)} distinct '
      f'joined keys exist;')
print(f'{len(shared)} keys are shared by exactly two sequences each, so {nonuniq} sequences '
      f'({100 * nonuniq / len(pairs):.1f}%) are not uniquely keyed')

[1, 3, 1] k=1 -> 0 (distances 0, 2, 2)
[1, 1, 1] k=2 -> 0
[1, 6, 1] k=3 -> 5

starting the search at lo = 1 differs on 316/5000 cases (6.3%)
  first one: nums=[0, 0, 0, 1, 4, 5, 6, 8], k=2 -> 1 instead of 0

[[3,2,1],[1,7,6],[2,7,7]] -> 1
[[1, 2, 34], [23, 5, 6], [4, 7, 8]]: correct 0, brute 0, joined-string key 1
of 9801 two-element sequences with values in 1..99, 9072 distinct joined keys exist;
729 keys are shared by exactly two sequences each, so 1458 sequences (14.9%) are not uniquely keyed


## Tests

Everything above, re-derived from scratch and asserted.

In [12]:
def check() -> None:
    """Everything the walkthrough claims, re-derived from scratch."""
    rng = np.random.default_rng(0)

    # The SVD route and the covariance route agree, up to the sign of each component.
    X = anisotropic(300, seed=0)
    p = PCA().fit(X)
    w, V = pca_via_covariance(X)
    assert np.allclose(p.explained_variance_, w, atol=1e-9)
    for j in range(2):
        assert abs(abs(float(p.components_[j] @ V[j])) - 1.0) < 1e-9
    # explained variance really is the variance of the projection
    Z = p.transform(X)
    assert np.allclose(Z.var(axis=0, ddof=1), p.explained_variance_, atol=1e-9)
    # and the components are orthonormal
    assert np.allclose(p.components_ @ p.components_.T, np.eye(2), atol=1e-9)

    # Without centring the first component chases the mean, and reports a HIGHER
    # explained-variance ratio for doing it.
    pu = PCA(center_data=False).fit(X)
    mean_dir = X.mean(axis=0) / np.linalg.norm(X.mean(axis=0))
    ang = lambda a, b: math.degrees(math.acos(min(1.0, abs(float(a @ b)))))
    assert ang(pu.components_[0], mean_dir) < 1.0
    assert ang(pu.components_[0], p.components_[0]) > 10.0
    assert pu.explained_variance_ratio_[0] > p.explained_variance_ratio_[0]

    # Units pick the axes.
    R = requests(300, seed=3)
    pr = PCA().fit(R)
    assert pr.explained_variance_ratio_[0] > 0.999
    assert abs(abs(pr.components_[0][0]) - 1.0) < 1e-6

    # A component can hold almost all of the variance and none of the signal.
    H, hy = hidden_signal(400, seed=1)
    ph = PCA().fit(H)
    Z2 = ph.transform(H, 2)
    a1 = best_threshold_accuracy(Z2[:, 0], hy)
    a2 = best_threshold_accuracy(Z2[:, 1], hy)
    assert ph.explained_variance_ratio_[0] > 0.9 and a1 < 0.65 < 0.95 < a2

    # Squaring the condition number destroys the small singular values.
    Xi = ill_conditioned(200, eps=1e-8, seed=0)
    sv = np.linalg.svd(Xi - Xi.mean(axis=0), full_matrices=False)[1]
    wi, _ = pca_via_covariance(Xi)
    assert sv[-1] > 1e-9 and np.sqrt(max(wi[-1], 0.0) * 199) == 0.0
    assert (pca_via_covariance(ill_conditioned(200, eps=1e-9, seed=0))[0] < 0).any()

    # Eckart-Young, exactly.
    A = rng.standard_normal((60, 20)) @ rng.standard_normal((20, 20))
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    for k in (1, 5, 10, 20):
        Ak = (U[:, :k] * s[:k]) @ Vt[:k]
        assert abs(float(np.linalg.norm(A - Ak)) - math.sqrt((s[k:] ** 2).sum())) < 1e-8

    # P is a symmetric distribution that sums to 1, and every row hits the perplexity
    # it was asked for.
    Xb, _ = blobs([1.0, 1.0, 1.0], [[0, 0, 0, 0, 0], [15, 0, 0, 0, 0], [0, 40, 0, 0, 0]],
                  60, seed=1)
    Pm, betas = joint_P(Xb, 30.0)
    assert abs(Pm.sum() - 1.0) < 1e-9 and np.abs(Pm - Pm.T).max() < 1e-15
    Dm = pairwise_sq_dists(Xb)
    Pc, _ = conditional_P(Dm, 30.0)
    for i in (0, 37, 90, 179):
        row = Pc[i][Pc[i] > 0]
        assert abs(2.0 ** float(-np.sum(row * np.log2(row))) - 30.0) < 0.1

    # The gradient really does decrease KL, and the map keeps neighbours.
    Yb, loss = tsne(Xb, perplexity=30.0, n_iter=600, lr=50.0, seed=0)
    assert loss[-1] < loss[0] / 4
    assert neighbour_recall(Xb, Yb, 10) > 0.5

    # Noise comes out looking clustered, and the clusters move when the seed does.
    Xn = pure_noise(180, 10, seed=7)
    s_raw = silhouette(Xn, kmeans_small(Xn, 3, seed=0))
    Yn, _ = tsne(Xn, perplexity=30.0, n_iter=600, lr=50.0, seed=0)
    assert silhouette(Yn, kmeans_small(Yn, 3, seed=0)) > 3 * s_raw
    runs_n = [tsne(Xn, perplexity=30.0, n_iter=600, lr=50.0, seed=sd)[0] for sd in range(2)]
    runs_b = [tsne(Xb, perplexity=30.0, n_iter=600, lr=50.0, seed=sd)[0] for sd in range(2)]
    assert pairwise_corr(*runs_n) < 0.6 < 0.95 < pairwise_corr(*runs_b)

    # Cluster width and between-cluster distance do not survive the trip.
    Xs, ys = blobs([0.5, 5.0, 0.5], [[0, 0, 0, 0, 0], [20, 0, 0, 0, 0], [0, 60, 0, 0, 0]],
                   60, seed=1)
    Ys_, _ = tsne(Xs, perplexity=30.0, n_iter=600, lr=50.0, seed=0)
    m = [ys == i for i in range(3)]
    o_sp = [spread(Xs[mi]) for mi in m]
    e_sp = [spread(Ys_[mi]) for mi in m]
    assert max(o_sp) / min(o_sp) > 10 and max(e_sp) / min(e_sp) < 1.5
    Xf, yf = blobs([1.0, 1.0, 1.0], [[0, 0, 0, 0, 0], [15, 0, 0, 0, 0], [0, 400, 0, 0, 0]],
                   60, seed=1)
    Yf, _ = tsne(Xf, perplexity=30.0, n_iter=600, lr=50.0, seed=0)
    mf = [yf == i for i in range(3)]
    assert centre_distance(Xf, mf[0], mf[2]) / centre_distance(Xf, mf[0], mf[1]) > 20
    assert centre_distance(Yf, mf[0], mf[2]) / centre_distance(Yf, mf[0], mf[1]) < 1.5

    # LeetCode 719: correct against brute force, and the lo = 1 slip is not.
    r719 = np.random.default_rng(3)
    diffs = 0
    for _ in range(2000):
        n = int(r719.integers(2, 9))
        a = [int(v) for v in r719.integers(0, 15, n)]
        k = int(r719.integers(1, n * (n - 1) // 2 + 1))
        want = kth_pair_distance_brute(a, k)
        assert SolutionKthPairDistance().smallestDistancePair(a[:], k) == want
        diffs += kth_pair_distance_lo1(a[:], k) != want
    assert diffs > 0
    assert SolutionKthPairDistance().smallestDistancePair([1, 3, 1], 1) == 0
    assert SolutionKthPairDistance().smallestDistancePair([1, 1, 1], 2) == 0
    assert SolutionKthPairDistance().smallestDistancePair([1, 6, 1], 3) == 5

    # LeetCode 2352: correct against brute force, and the joined-string key is not.
    r2352 = np.random.default_rng(4)
    for _ in range(1500):
        n = int(r2352.integers(1, 5))
        g = [[int(v) for v in r2352.integers(1, 60, n)] for _ in range(n)]
        assert SolutionEqualPairs().equalPairs(g) == equal_pairs_brute(g)
    assert SolutionEqualPairs().equalPairs([[3, 2, 1], [1, 7, 6], [2, 7, 7]]) == 1
    assert SolutionEqualPairs().equalPairs([[3, 1, 2, 2], [1, 4, 4, 5], [2, 4, 2, 2],
                                            [2, 4, 2, 2]]) == 3
    trap = [[1, 2, 34], [23, 5, 6], [4, 7, 8]]
    assert SolutionEqualPairs().equalPairs(trap) == 0 == equal_pairs_brute(trap)
    assert equal_pairs_joined(trap) == 1


check()
print('all assertions passed')

all assertions passed
